# Frequency Response Example

Quick demonstration of vessel response changing measured probe angle.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import shutil
import xarray as xr
import numpy as np
from OpenFUSIONToolkit._core import OFT_env
from synthwave import PACKAGE_ROOT
from synthwave.magnetic_geometry.utils import create_torus_mesh
from synthwave.magnetic_geometry.filaments import ToroidalFilamentTracer
from synthwave.mirnov.prep_thincurr_input import (
    gen_OFT_filament_and_eta_file,
    gen_OFT_sensors_file,
)
from synthwave.mirnov.run_thincurr_model import calc_frequency_response

EXAMPLE_DIR = os.path.join(PACKAGE_ROOT, "..", "examples", "freq_response")

MAJOR_RADIUS = 1
MINOR_RADIUS = 0.5

os.makedirs(EXAMPLE_DIR, exist_ok=True)

torus_mesh_file = os.path.join(EXAMPLE_DIR, "thincurr_torus_mesh.h5")
if not os.path.exists(torus_mesh_file):
    torus_mesh = create_torus_mesh(MAJOR_RADIUS, MINOR_RADIUS)
    torus_mesh.write_to_file(torus_mesh_file)

# Create 'oft_in.xml' file with icoil definitions
toroidal_tracer = ToroidalFilamentTracer(2, 1, MAJOR_RADIUS, 0, MINOR_RADIUS - 0.1)
filament_list, _ = toroidal_tracer.get_filament_list(num_filaments=10)

gen_OFT_filament_and_eta_file(
    working_directory=EXAMPLE_DIR,
    filament_list=filament_list,
    resistivity_list=[1e-6],
)

sensor_details = xr.Dataset(
    data_vars={
        "position": (
            ("sensor_idx", "coord"),
            np.array(
                [
                    [MAJOR_RADIUS + MINOR_RADIUS, 0.0, 0.0],  # sensor on x axis
                    [0, MAJOR_RADIUS + MINOR_RADIUS, 0.0],  # sensor on y axis
                    [MAJOR_RADIUS, 0, MINOR_RADIUS],  # sensor on z axis
                ]
            ),
        ),
        "normal": (
            ("sensor_idx", "coord"),
            np.array(
                [
                    [1.0, 0.0, 0.0],
                    [0.0, 1.0, 0.0],
                    [0.0, 0.0, 1.0],
                ]
            ),
        ),
        "radius": ("sensor_idx", np.array([0.01, 0.01, 0.01])),
    },
    coords={
        "sensor_idx": np.array(["sensor_x", "sensor_y", "sensor_z"]),
    },
    attrs={
        "sensor_set_name": "test_sensors",
    },
)
sensor_file_path = gen_OFT_sensors_file(
    sensor_details=sensor_details,
    working_directory=EXAMPLE_DIR,
)

oft_tmpdir = os.path.join("tmp", f"oft_{os.getpid()}")
if os.path.exists(oft_tmpdir):
    shutil.rmtree(oft_tmpdir)

oft_env = OFT_env(nthreads=min(4, os.cpu_count()))

In [ ]:
total_response, direct_response, vessel_response = calc_frequency_response(
    oft_env=oft_env,
    tracer=toroidal_tracer,
    freq=10e3,
    working_directory=EXAMPLE_DIR,
    mesh_file=torus_mesh_file,
    sensor_file_path=sensor_file_path,
    sensor_details=sensor_details,
)

# Get the angle of each response
total_response_angle = np.angle(total_response)
vessel_response_angle = np.angle(vessel_response)
direct_response_angle = np.angle(direct_response)

# Print the results
for i, sensor in enumerate(sensor_details.sensor_idx.values):
    print(f"Sensor: {sensor}")
    print(f"  Total Response Angle: {total_response_angle[i]:.2f} rad")
    print(f"  Vessel Response Angle: {vessel_response_angle[i]:.2f} rad")
    print(f"  Direct Response Angle: {direct_response_angle[i]:.2f} rad")